# How Toxic Is Your Flow? — Companion Notebook

Reproduces the analysis from the [blog post](https://jgehunter.github.io/microstructure/how-toxic-is-your-flow/).
Requires: `pip install markoutlib`

In [ ]:
from datetime import datetime, timedelta

import polars as pl
from fetch_data import fetch_binance_aggtrades, prepare_binance_trades_and_quotes

import markoutlib as mo

## Load data and define windows

In [ ]:
raw = fetch_binance_aggtrades("BTCUSDT", "2026-02-28")
trades, quotes = prepare_binance_trades_and_quotes(raw)

pre_end = datetime(2026, 2, 28, 5, 0, 0)
post_start = datetime(2026, 2, 28, 5, 30, 0)
post_end = datetime(2026, 2, 28, 11, 0, 0)

trades_pre = trades.filter(pl.col("timestamp") < pre_end)
quotes_pre = quotes.filter(pl.col("timestamp") < pre_end + pl.duration(minutes=10))
trades_post = trades.filter(
    (pl.col("timestamp") >= post_start) & (pl.col("timestamp") < post_end)
)
quotes_post = quotes.filter(
    (pl.col("timestamp") >= post_start)
    & (pl.col("timestamp") < post_end + pl.duration(minutes=10))
)

print(f"Pre-strike:  {trades_pre.shape[0]:,} trades")
print(f"Post-strike: {trades_post.shape[0]:,} trades")

## Markout curves: pre vs post strike

In [ ]:
result_pre = mo.compute(
    trades=trades_pre,
    quotes=quotes_pre,
    horizons=mo.seconds(1, 5, 10, 30, 60, 300),
    unit="bps",
    perspective="maker",
)
result_post = mo.compute(
    trades=trades_post,
    quotes=quotes_post,
    horizons=mo.seconds(1, 5, 10, 30, 60, 300),
    unit="bps",
    perspective="maker",
)

print("Pre-strike:")
print(result_pre.curve())
print("\nPost-strike:")
print(result_post.curve())

## Size segmentation (post-strike)

In [ ]:
q33 = trades_post["size"].quantile(0.33)
q67 = trades_post["size"].quantile(0.67)

trades_tagged = trades_post.with_columns(
    pl.when(pl.col("size") <= q33)
    .then(pl.lit("small"))
    .when(pl.col("size") <= q67)
    .then(pl.lit("medium"))
    .otherwise(pl.lit("large"))
    .alias("size_bucket")
)

result_sized = mo.compute(
    trades=trades_tagged,
    quotes=quotes_post,
    horizons=mo.seconds(1, 5, 10, 30, 60, 300),
    unit="bps",
    perspective="maker",
)
result_sized.plot.curve(by="size_bucket")

## Spread decomposition

In [ ]:
print("Pre-strike:")
for h in [1, 5, 10, 30]:
    print(result_pre.spread_decomposition(horizon=mo.seconds(h)))

print("\nPost-strike:")
for h in [1, 5, 10, 30]:
    print(result_post.spread_decomposition(horizon=mo.seconds(h)))

## BTC vs DOGE: trade-clock comparison

In [ ]:
raw_btc_mar = fetch_binance_aggtrades("BTCUSDT", "2026-03-01")
trades_btc, _ = prepare_binance_trades_and_quotes(raw_btc_mar)

raw_doge = fetch_binance_aggtrades("DOGEUSDT", "2026-03-01")
trades_doge, _ = prepare_binance_trades_and_quotes(raw_doge)

r_btc = mo.compute(
    trades_btc,
    horizons=mo.trades(1, 5, 10, 25, 50, 100),
    unit="bps",
    perspective="maker",
)
r_doge = mo.compute(
    trades_doge,
    horizons=mo.trades(1, 5, 10, 25, 50, 100),
    unit="bps",
    perspective="maker",
)

print("BTC trade-clock:")
print(r_btc.curve())
print("\nDOGE trade-clock:")
print(r_doge.curve())

## Information leakage: crossing-zero curve

In [ ]:
leak_end = datetime(2026, 2, 28, 7, 30, 0)
trades_leak = trades_post.filter(pl.col("timestamp") < leak_end)
quotes_leak = quotes_post.filter(
    pl.col("timestamp") < leak_end + pl.duration(minutes=5)
)

result_leak = mo.compute(
    trades=trades_leak,
    quotes=quotes_leak,
    horizons=mo.seconds_range(-30, 30, 1),
    unit="bps",
    perspective="maker",
)
result_leak.plot.curve()

## Pre-strike flow analysis

In [ ]:
print(f"{'Window':<16} {'Trades':>8} {'Sell%':>7} {'Imbalance':>11}")
for start_min in range(120, 360, 15):
    h, m = divmod(start_min, 60)
    t0 = datetime(2026, 2, 28, h, m)
    t1 = t0 + timedelta(minutes=15)
    w = trades.filter((pl.col("timestamp") >= t0) & (pl.col("timestamp") < t1))
    n = w.shape[0]
    if n < 50:
        continue
    sf = (w["side"] == -1).sum() / n * 100
    net = (w["side"] * w["size"] * w["price"]).sum()
    tot = (w["size"] * w["price"]).sum()
    imb = net / tot * 100
    eh, em = divmod(start_min + 15, 60)
    print(f"  {h:02d}:{m:02d}-{eh:02d}:{em:02d}  {n:>8,}  {sf:>5.1f}%  {imb:>+8.1f}%")

## Markout of pre-strike sellers

In [ ]:
for label, t0, t1 in [
    ("04:00-04:30", datetime(2026, 2, 28, 4, 0), datetime(2026, 2, 28, 4, 30)),
    ("05:15-05:30", datetime(2026, 2, 28, 5, 15), datetime(2026, 2, 28, 5, 30)),
]:
    tw = trades.filter((pl.col("timestamp") >= t0) & (pl.col("timestamp") < t1))
    qw = quotes.filter(pl.col("timestamp") < datetime(2026, 2, 28, 7, 0))
    n = tw.shape[0]
    sf = (tw["side"] == -1).sum() / n * 100
    print(f"\n{label}: {n:,} trades, {sf:.1f}% sells")
    r = mo.compute(
        tw,
        qw,
        horizons=mo.seconds(60, 300, 1800, 3600),
        unit="bps",
        perspective="taker",
    )
    print(r.curve().select("horizon_value", "markout_mean", "t_stat"))